Testing calculations on punctuality and cancellations

In [21]:
import pandas as pd
import numpy as np

In [22]:
try: 
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet("intermediate_outputs/data_chuuchuu_combined_operators.parquet")

In [23]:
cancelled_t = len(data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="t"])/len(data_chuuchuu["arrivalCancelled"])*100
cancelled_f = len(data_chuuchuu[data_chuuchuu["arrivalCancelled"]=="f"])/len(data_chuuchuu["arrivalCancelled"])*100

print("Share of row arrivalCancelled = t")
print(round(cancelled_t,2),"%")
print("Share of row arrivalCancelled = f")
print(round(cancelled_f,2),"%")
print("Share of row without data cancelation")
print(round(100-(cancelled_f+cancelled_t),2),"%")

Share of row arrivalCancelled = t
2.8 %
Share of row arrivalCancelled = f
78.4 %
Share of row without data cancelation
18.8 %


### Recovering a cancellation status for rows where `arrivalCancelled` is null

Per the terminology doc: *"If null and a delay was recorded, you can assume the arrival was not cancelled."*

Two ways to check "was something recorded": whether `arrivalDelay` is non-null, or whether the effective `arrival` timestamp itself is non-null. These aren't the same thing -- `arrivalDelay` also needs `plannedArrival` to be present to be computable, so a row can have a real recorded `arrival` time with a still-`NaN` `arrivalDelay` (missing `plannedArrival`). Checking `arrival` directly is a strict superset of the delay-based check and matches the actual question ("did the train effectively arrive?") more directly, so that's the criterion used below.

In [24]:
null_cancelled = data_chuuchuu["arrivalCancelled"].isna()

# among the rows with no explicit flag, an effective arrival timestamp means it wasn't cancelled
inferred_not_cancelled = null_cancelled & data_chuuchuu["arrival"].notna()

# still no way to know -- neither an explicit flag nor an effective arrival to infer from
unreliable = null_cancelled & data_chuuchuu["arrival"].isna()

n = len(data_chuuchuu)
share_explicit = (~null_cancelled).sum() / n * 100
share_inferred = inferred_not_cancelled.sum() / n * 100
share_unreliable = unreliable.sum() / n * 100

print(f"Share of rows with an explicit arrivalCancelled value (t/f): {share_explicit:.2f}%")
print(f"Share of rows with arrivalCancelled null but inferred not cancelled (arrival is not null): {share_inferred:.2f}%")
print(f"Share of rows with no reliable cancellation data at all: {share_unreliable:.2f}%")
print()
print(f"Total reliable cancellation data: {share_explicit + share_inferred:.2f}%")

Share of rows with an explicit arrivalCancelled value (t/f): 81.20%
Share of rows with arrivalCancelled null but inferred not cancelled (arrival is not null): 17.65%
Share of rows with no reliable cancellation data at all: 1.16%

Total reliable cancellation data: 98.84%


In [25]:
data_chuuchuu["arrivalCancelled_resolved"] = data_chuuchuu["arrivalCancelled"]
data_chuuchuu.loc[inferred_not_cancelled, "arrivalCancelled_resolved"] = "f"

data_chuuchuu["arrivalCancelled_resolved"].value_counts(dropna=False)

arrivalCancelled_resolved
f       14407373
t         419652
None      173589
Name: count, dtype: int64

In [26]:
unreliable_rows = data_chuuchuu[unreliable]
unreliable_rows

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,sort_time_source,journey_id,is_ambiguous_trip,journey_verificator,is_cross_agency_duplicate,cross_agency_duplicate_confidence,depart_terminus,journey_type,normalized_operator,arrivalCancelled_resolved
2891,IT,REG,801,2026-02-26,8302988,2026-02-26 00:10:40.757+00,REG 801,S01066,MILANO CADORNA,NaT,...,departure,IT_REG_801_2026-02-26,False,REG_801_2026-02-26_8302988,False,not_flagged,depart,domestic,Trenitalia,None
2902,DB,IR,3393,2026-02-26,8507000,2026-02-26 00:11:47.283+00,IR 3393,8507000,Bern,NaT,...,departure,DB_IR_3393_2026-02-26,False,IR_3393_2026-02-26_8507000,False,not_flagged,depart,domestic,None,None
2911,IT,REG,701,2026-02-26,8302988,2026-02-26 00:15:56.35+00,REG 701,S01066,MILANO CADORNA,NaT,...,departure,IT_REG_701_2026-02-26,False,REG_701_2026-02-26_8302988,False,not_flagged,depart,domestic,Trenitalia,None
3622,DB,IR,1842,2026-02-26,8501303,2026-02-26 00:20:56.635+00,IR 1842,8501303,Villeneuve(CH),NaT,...,departure,DB_IR_1842_2026-02-26,False,IR_1842_2026-02-26_8501303,False,not_flagged,depart,domestic,None,None
3630,IT,REG,25091,2026-02-26,8300066,2026-02-26 00:22:06.689+00,REG 25091,S01318,SEREGNO,NaT,...,departure,IT_REG_25091_2026-02-26,False,REG_25091_2026-02-26_8300066,False,not_flagged,depart,domestic,Trenitalia,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15000497,OEBB,NJ,493,2026-06-17,8000147,2026-06-18 11:58:55+00,NJ 493,8000147,Hamburg-Harburg,NaT,...,departure,OEBB_NJ_493_2026-06-17,False,NJ_493_2026-06-17_8000147,True,likely_true_duplicate,intermediate,international,OEBB,None
15000513,DB,NJ,493,2026-06-17,8002553,2026-06-18 11:59:16.845+00,NJ 493,8002553,Hamburg-Altona,NaT,...,departure,DB_NJ_493_2026-06-17,False,NJ_493_2026-06-17_8002553,True,likely_true_duplicate,depart,international,OEBB,None
15000529,DB,NJ,40421,2026-06-17,8400058,2026-06-18 12:00:11.939+00,NJ 40421,8400058,Amsterdam Centraal,NaT,...,departure,DB_NJ_40421_2026-06-17,False,NJ_40421_2026-06-17_8400058,True,needs_review,depart,international,OEBB,None
15000540,IT,ICN,89512,2026-06-17,8300337,2026-06-18 13:24:46.052+00,ICN 89512,S11781,REGGIO DI CALABRIA CENTRALE,NaT,...,departure,IT_ICN_89512_2026-06-17,False,ICN_89512_2026-06-17_8300337,False,not_flagged,depart,domestic,Trenitalia,None


### Checking the literal terminology-doc criterion: "if null and a delay was recorded, assume not cancelled"

This applies the rule exactly as written (`arrivalDelay` non-null), rather than the `arrival`-based criterion used above, so the two can be compared directly.

In [27]:
inferred_not_cancelled_delay = null_cancelled & data_chuuchuu["arrivalDelay"].notna()
unreliable_delay = null_cancelled & data_chuuchuu["arrivalDelay"].isna()

share_inferred_delay = inferred_not_cancelled_delay.sum() / n * 100
share_unreliable_delay = unreliable_delay.sum() / n * 100

print(f"Share of rows with arrivalCancelled null but inferred not cancelled (arrivalDelay is not null): {share_inferred_delay:.2f}%")
print(f"Share of rows with no reliable cancellation data at all (delay-based criterion): {share_unreliable_delay:.2f}%")
print()
print(f"Total reliable cancellation data (delay-based criterion): {share_explicit + share_inferred_delay:.2f}%")

Share of rows with arrivalCancelled null but inferred not cancelled (arrivalDelay is not null): 17.52%
Share of rows with no reliable cancellation data at all (delay-based criterion): 1.28%

Total reliable cancellation data (delay-based criterion): 98.72%


In [28]:
# rows where the arrival-based and delay-based criteria disagree (both only defined among null_cancelled rows)
criteria_disagree = inferred_not_cancelled != inferred_not_cancelled_delay
print(f"{criteria_disagree.sum()} rows where the two criteria disagree")

# by construction (arrivalDelay needs plannedArrival + arrival to both be present), the delay-based
# criterion can only be a subset of the arrival-based one -- confirm that's actually the case here
only_arrival_based = inferred_not_cancelled & ~inferred_not_cancelled_delay
only_delay_based = inferred_not_cancelled_delay & ~inferred_not_cancelled
print(f"rows caught by arrival-based but not delay-based: {only_arrival_based.sum()}")
print(f"rows caught by delay-based but not arrival-based: {only_delay_based.sum()} (expected to be 0)")

data_chuuchuu[only_arrival_based][["arrival", "plannedArrival", "arrivalDelay", "arrivalCancelled"]].head(20)

18580 rows where the two criteria disagree
rows caught by arrival-based but not delay-based: 18580
rows caught by delay-based but not arrival-based: 0 (expected to be 0)


,arrival,plannedArrival,arrivalDelay,arrivalCancelled
106930,2026-02-26 05:43:00+00:00,2026-02-26 05:43:00+00:00,NaN,None
106931,2026-02-26 05:31:00+00:00,2026-02-26 05:31:00+00:00,NaN,None
106933,2026-02-26 05:04:00+00:00,2026-02-26 05:04:00+00:00,NaN,None
207344,2026-02-26 04:53:00+00:00,2026-02-26 04:53:00+00:00,NaN,None
207346,2026-02-26 05:25:00+00:00,2026-02-26 05:25:00+00:00,NaN,None
207348,2026-02-26 06:00:00+00:00,2026-02-26 06:00:00+00:00,NaN,None
207351,2026-02-26 06:19:00+00:00,2026-02-26 06:19:00+00:00,NaN,None
207352,2026-02-26 06:29:00+00:00,2026-02-26 06:29:00+00:00,NaN,None
219348,2026-02-26 06:10:00+00:00,2026-02-26 06:10:00+00:00,NaN,None
220381,2026-02-26 07:10:00+00:00,2026-02-26 07:10:00+00:00,NaN,None


### Country ranking: domestic punctuality at the terminus (5-minute threshold)

Restricted to `journey_type == "domestic"` and `depart_terminus == "terminus"` -- the last stop of each resolved trip, so each row here is one journey's actual outcome rather than a per-stop measurement.

A terminus row is **assessable** only if its cancellation status is reliable (`arrivalCancelled_resolved` is not null) and either it was cancelled (automatically counted as not punctual) or its `arrivalDelay` is known. Rows failing that -- unreliable cancellation status, or not cancelled but with an unknown delay (the residual edge case identified above) -- are excluded rather than guessed at.

**Punctual** = not cancelled and `arrivalDelay <= 5` (minutes).

Countries with very few `n_terminus_arrivals` should be read with caution -- their rate is based on too few journeys to be meaningful.

In [29]:
scope = (data_chuuchuu["journey_type"] == "domestic") & (data_chuuchuu["depart_terminus"] == "terminus")

reliable_cancellation = data_chuuchuu["arrivalCancelled_resolved"].notna()
is_cancelled = data_chuuchuu["arrivalCancelled_resolved"] == "t"
has_delay = data_chuuchuu["arrivalDelay"].notna()

# assessable for punctuality: cancellation status is known, and either it was cancelled (automatically
# not punctual) or we have an actual delay to check against the 5-minute threshold
assessable = reliable_cancellation & (is_cancelled | has_delay)
punctual_5min = assessable & ~is_cancelled & (data_chuuchuu["arrivalDelay"] <= 300)#5 min = 300 seconds

mask = scope & assessable
country_punctuality = pd.DataFrame({
    "country": data_chuuchuu.loc[mask, "country"],
    "punctual": punctual_5min[mask],
})

country_ranking = country_punctuality.groupby("country").agg(
    n_terminus_arrivals=("punctual", "size"),
    n_punctual=("punctual", "sum"),
)
country_ranking["punctuality_rate_pct"] = (country_ranking["n_punctual"] / country_ranking["n_terminus_arrivals"] * 100).round(2)
country_ranking = country_ranking.sort_values("punctuality_rate_pct", ascending=False)

print(f"{(scope & ~assessable).sum()} domestic terminus rows excluded as not assessable (unreliable cancellation status or unknown delay)")
print()
country_ranking

6651 domestic terminus rows excluded as not assessable (unreliable cancellation status or unknown delay)



,n_terminus_arrivals,n_punctual,punctuality_rate_pct
country,,,
Czech Republic,9,9,100.00
Romania,2,2,100.00
Switzerland,125545,123146,98.09
Netherlands,78464,74500,94.95
Belgium,40229,38134,94.79
Denmark,43690,40013,91.58
Austria,20981,18359,87.50
France,114470,99212,86.67
Italy,123294,106749,86.58
